In [ ]:
from pathlib import Path
import sys
import os
import random
import importlib



from pathlib import Path
from rasterio import features
import numpy as np
import os
import pandas as pd
import random
import sys
import satellites.shared.io as io_l
from satellites.ml_classification import ClassificationPipelineConfig
import satellites.ml_classification.sensitivity.runner as sensitivity_runner_module

sensitivity_runner_module = importlib.reload(sensitivity_runner_module)
ClassificationSensitivityRunner = sensitivity_runner_module.ClassificationSensitivityRunner

# Global reproducibility seed for notebook-side random operations
random_seed = 42
os.environ["PYTHONHASHSEED"] = str(random_seed)
os.environ["TF_DETERMINISTIC_OPS"] = "1"
random.seed(random_seed)
np.random.seed(random_seed)

In [2]:
sample_gdf_file = "E:/Eleni/git_repo/work_dir/test/input_gdf.gpkg"
gdf_sample = io_l.read_data(sample_gdf_file)
gdf_sample = gdf_sample.set_crs("EPSG:2100", allow_override=True)
gdf_sample = gdf_sample[gdf_sample["label"].notna()].reset_index(drop=True)
gdf_train = gdf_sample.copy()

unknown_fraction = 0.20
rng = np.random.default_rng(random_seed)
unknown_count = int(len(gdf_train) * unknown_fraction)
unknown_indices = rng.choice(gdf_train.index.to_numpy(), size=unknown_count, replace=False)
gdf_train.loc[unknown_indices, "label"] = None

features = [col for col in gdf_sample.columns if col not in ["label", "geometry"]]
print(f"Rows: {len(gdf_sample)} | Unknown labels injected: {gdf_train['label'].isna().sum()}")

Reading file: E:/Eleni/git_repo/work_dir/test/input_gdf.gpkg
Rows: 10000 | Unknown labels injected: 2000


In [3]:
# Sensitivity setup
sensitivity_output_dir = Path("E:/Eleni/git_repo/work_dir/test/test_new_ml_sensitivity")
sensitivity_output_dir.mkdir(parents=True, exist_ok=True)

seed_values = [11, 22, 33, 42]

def build_config_for_seed(seed: int) -> ClassificationPipelineConfig:
    project_dir = sensitivity_output_dir / f"run_seed_{seed}"
    return ClassificationPipelineConfig(
        project_dir=project_dir,
        target_column="label",
        feature_columns=features,
        id_column="row_id",
        cv_folds=3,
        test_size=0.2,
        random_state=seed,
        n_jobs=1,
        max_search_candidates=12,
        selection_type="soft_voting",
        interpretability_top_models=3,
        interpretability_include_shap=False,
        top_voting_models=None,
        # label_balancing_method="random_oversample",
        spatial_split=True,
        spatial_split_grid_size=10,
        selected_models=(
            "gradient_boosting",
            "hist_gradient_boosting",
            "bagging",
            "random_forest",
            "decision_tree",
            "extra_trees",
            "bayesian",
            "xgboost",
            #"lightgbm",
            "knn",
            "neural_network"
    ),
        max_map_geometries=20000,
    )

In [4]:
runner = ClassificationSensitivityRunner(output_dir=sensitivity_output_dir)

# Option A (all-in-one):
# results_df = runner.run(gdf=gdf_train, seeds=seed_values, config_builder=build_config_for_seed)

# Option B (step 1): run all seeds and save only the CSV summary
results_df = runner.run_seeds(
    gdf=gdf_train,
    seeds=seed_values,
    config_builder=build_config_for_seed,
    write_csv=True,
)

results_df

Running seed: 11...


E:\Eleni\git_repo\gis_repo\src\ml_classification_pipeline\core\config.py:161: UserWarning: feature_importance_top_n=15 is larger than the number of feature_columns (7). Using 7 instead.
  self.feature_importance_top_n = self._cap_with_warning(



Function <run_check> started on 2026-07-04 19:47:43

Checking input data...

Resetting project output folder: E:\Eleni\git_repo\work_dir\test\test_new_ml_sensitivity\run_seed_11
Folder 'E:/Eleni/git_repo/work_dir/test/test_new_ml_sensitivity/run_seed_11/01_check' exists. Deleting...
Folder 'E:/Eleni/git_repo/work_dir/test/test_new_ml_sensitivity/run_seed_11/01_check' deleted successfully.
Folder 'E:/Eleni/git_repo/work_dir/test/test_new_ml_sensitivity/run_seed_11/02_prepare' exists. Deleting...
Folder 'E:/Eleni/git_repo/work_dir/test/test_new_ml_sensitivity/run_seed_11/02_prepare' deleted successfully.
Folder 'E:/Eleni/git_repo/work_dir/test/test_new_ml_sensitivity/run_seed_11/03_train' exists. Deleting...
Folder 'E:/Eleni/git_repo/work_dir/test/test_new_ml_sensitivity/run_seed_11/03_train' deleted successfully.
Folder 'E:/Eleni/git_repo/work_dir/test/test_new_ml_sensitivity/run_seed_11/04_evaluate' exists. Deleting...
Folder 'E:/Eleni/git_repo/work_dir/test/test_new_ml_sensitivity/ru

c:\ProgramData\anaconda3\envs\axhub_spatial\Lib\site-packages\sklearn\ensemble\_bagging.py:98: UserWarning: Using the fractional value max_samples=0.5 when the number of samples is 16 results in a low number (8) of bootstrap samples. We recommend passing `max_samples` as an integer instead.
  warn(
c:\ProgramData\anaconda3\envs\axhub_spatial\Lib\site-packages\sklearn\ensemble\_bagging.py:98: UserWarning: Using the fractional value max_samples=0.5 when the number of samples is 15 results in a low number (7) of bootstrap samples. We recommend passing `max_samples` as an integer instead.
  warn(
c:\ProgramData\anaconda3\envs\axhub_spatial\Lib\site-packages\sklearn\ensemble\_bagging.py:98: UserWarning: Using the fractional value max_samples=0.5 when the number of samples is 16 results in a low number (8) of bootstrap samples. We recommend passing `max_samples` as an integer instead.
  warn(
c:\ProgramData\anaconda3\envs\axhub_spatial\Lib\site-packages\sklearn\ensemble\_bagging.py:98: UserW


Saved artifacts for bagging: model=E:\Eleni\git_repo\work_dir\test\test_new_ml_sensitivity\run_seed_11\03_train\models\bagging\best_model.joblib, cv_results=E:\Eleni\git_repo\work_dir\test\test_new_ml_sensitivity\run_seed_11\03_train\models\bagging\cv_results.csv, plot=E:\Eleni\git_repo\work_dir\test\test_new_ml_sensitivity\run_seed_11\03_train\models\bagging\search_results.png

Model bagging finished in duration: 00:00:13
____________________________________________________________________________________________________

Training model bayesian


...

c:\ProgramData\anaconda3\envs\axhub_spatial\Lib\site-packages\sklearn\ensemble\_bagging.py:98: UserWarning: Using the fractional value max_samples=0.5 when the number of samples is 16 results in a low number (8) of bootstrap samples. We recommend passing `max_samples` as an integer instead.
  warn(
c:\ProgramData\anaconda3\envs\axhub_spatial\Lib\site-packages\sklearn\ensemble\_bagging.py:98: UserWarning: Using the fractional value max_samples=0.5 when the number of samples is 15 results in a low number (7) of bootstrap samples. We recommend passing `max_samples` as an integer instead.
  warn(
c:\ProgramData\anaconda3\envs\axhub_spatial\Lib\site-packages\sklearn\ensemble\_bagging.py:98: UserWarning: Using the fractional value max_samples=0.5 when the number of samples is 16 results in a low number (8) of bootstrap samples. We recommend passing `max_samples` as an integer instead.
  warn(
c:\ProgramData\anaconda3\envs\axhub_spatial\Lib\site-packages\sklearn\ensemble\_bagging.py:98: UserW


Saved artifacts for bagging: model=E:\Eleni\git_repo\work_dir\test\test_new_ml_sensitivity\run_seed_42\03_train\models\bagging\best_model.joblib, cv_results=E:\Eleni\git_repo\work_dir\test\test_new_ml_sensitivity\run_seed_42\03_train\models\bagging\cv_results.csv, plot=E:\Eleni\git_repo\work_dir\test\test_new_ml_sensitivity\run_seed_42\03_train\models\bagging\search_results.png

Model bagging finished in duration: 00:00:09
____________________________________________________________________________________________________

Training model bayesian


c:\ProgramData\anaconda3\envs\axhub_spatial\Lib\site-packages\sklearn\model_selection\_search.py:324: UserWarning: The total space of parameters 4 is smaller than n_iter=12. Running 4 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(



Saved artifacts for bayesian: model=E:\Eleni\git_repo\work_dir\test\test_new_ml_sensitivity\run_seed_42\03_train\models\bayesian\best_model.joblib, cv_results=E:\Eleni\git_repo\work_dir\test\test_new_ml_sensitivity\run_seed_42\03_train\models\bayesian\cv_results.csv, plot=E:\Eleni\git_repo\work_dir\test\test_new_ml_sensitivity\run_seed_42\03_train\models\bayesian\search_results.png

Model bayesian finished in duration: 00:00:00
____________________________________________________________________________________________________

Training model hist_gradient_boosting

Saved artifacts for hist_gradient_boosting: model=E:\Eleni\git_repo\work_dir\test\test_new_ml_sensitivity\run_seed_42\03_train\models\hist_gradient_boosting\best_model.joblib, cv_results=E:\Eleni\git_repo\work_dir\test\test_new_ml_sensitivity\run_seed_42\03_train\models\hist_gradient_boosting\cv_results.csv, plot=E:\Eleni\git_repo\work_dir\test\test_new_ml_sensitivity\run_seed_42\03_train\models\hist_gradient_boosting\sear

,seed,success,selected_model,test_accuracy,project_dir,error
0,11,True,soft_voting,0.9925000000,E:\Eleni\git_rep...,None
1,22,True,soft_voting,0.9912500000,E:\Eleni\git_rep...,None
2,33,True,soft_voting,0.9950000000,E:\Eleni\git_rep...,None
3,42,True,soft_voting,0.9981250000,E:\Eleni\git_rep...,None


In [5]:
# Option B (step 2): create plot + HTML report from an existing results table
results_df = pd.read_csv(sensitivity_output_dir / "sensitivity_results.csv")
artifacts = runner.create_report(results_df, write_csv=False)
artifacts

{'csv_path': 'E:\\Eleni\\git_repo\\work_dir\\test\\test_new_ml_sensitivity\\sensitivity_results.csv',
 'plot_path': 'E:\\Eleni\\git_repo\\work_dir\\test\\test_new_ml_sensitivity\\accuracy_by_seed.png',
 'report_path': 'E:\\Eleni\\git_repo\\work_dir\\test\\test_new_ml_sensitivity\\sensitivity_report.html'}

In [6]:
import webbrowser
html_path = sensitivity_output_dir / "sensitivity_report.html"
webbrowser.open(html_path.as_uri())

True